# Unihra SDK: Complete SEO Content Analysis

> **Professional guide for comparing your pages against competitors and getting actionable SEO recommendations.**

<div align="center">

| What you'll learn | Sections |
|---|---|
| Setup & auth | §1 |
| Run full analysis | §2 |
| Semantic gaps (add to H1 / Title / body) | §3 |
| Anchors & link texts (with `href` values) | §4 |
| TF‑IDF word actions | §5 |
| Phrases (n‑grams) | §6 |
| Triplets — Knowledge Graph (extended mode) | §7 |
| Page structure & heading hierarchy | §8 |
| Export to Excel | §9 |

</div>

## What Unihra does

Unihra crawls your URL and each competitor URL on the server side, then returns:
- **What is missing or weak** on your page vs. competitors
- **Where** to fix it (title, H1–H3, body text, anchor links)
- **Exact words/phrases** with priority scores and real-usage snippets
- **Knowledge Graph of facts** (subjects → predicates → objects) when `triplet_analysis=True`

> 💳 Cost: standard analysis = **1 credit**, extended analysis with triplets (`triplet_analysis=True`) = **5 credits**.

The API key can be obtained via Telegram bot: [@UniHRA_bot](https://t.me/UniHRA_bot).

In [ ]:
#  ͟͟͟͟Install dependencies (first run only)͟͟͟͟
# pip install "unihra[full]"  # full = pandas + openpyxl + tqdm + report export

## § 1 — Setup & Authentication

In [ ]:
import os
import pandas as pd
from IPython.display import display, Markdown
from unihra import UnihraClient

# ─── Paste your key here or use the UNIHRA_API_KEY env var ───
API_KEY = os.getenv("UNIHRA_API_KEY", "YOUR_API_KEY_HERE")

# Initialise the client with automatic HTTP retries
client = UnihraClient(api_key=API_KEY, max_retries=3)

## § 2 — Define pages & run analysis

We use **real-world sites** so you see meaningful results right away. Replace them with your own URLs when ready.

Set `ENABLE_TRIPLETS = True` below to also extract a Knowledge Graph (subject → predicate → object facts) from competitor pages. That mode costs **5 credits** instead of **1**, so leave it `False` unless you specifically need fact-coverage analysis.

In [ ]:
# ─── Target page (yours) ───
own_page = "https://www.apple.com/iphone-16/"

# ─── Competitors (3 – 5 is a good baseline) ───
competitors = [
    "https://www.samsung.com/us/smartphones/galaxy-s25/",
    "https://store.google.com/product/pixel_9",
    "https://www.oneplus.com/us/oneplus-13",
]

# ─── Queries you want the page to rank for ───
target_queries = [
    "buy smartphone online",
    "best camera phone 2025",
]

# ─── Optional: cookies for pages behind walls ───
cookies_config = {}  # e.g. {own_page: "session=abc; region=us"}

# ─── Knowledge Graph mode (5 credits instead of 1) ───
ENABLE_TRIPLETS = True   # set False to run the cheap standard analysis

print(f"🚀  Starting analysis for:\n   {own_page}")
print(f"🔎  Target queries:  {target_queries}")
print(f"🧠  Triplets / Knowledge Graph: {'ON (5 credits)' if ENABLE_TRIPLETS else 'OFF (1 credit)'}")

In [ ]:
result = client.analyze(
    own_page=own_page,
    competitors=competitors,
    queries=target_queries,
    lang="en",
    url_cookies=cookies_config,
    triplet_analysis=ENABLE_TRIPLETS,
    verbose=True,      # shows a live progress bar
)
display(Markdown("✅  **Analysis complete!**  Keys returned: `" + "`, `".join(result.keys()) + "`"))

## § 3 — Semantic Gaps  *(Priority content to add)*

Each row tells you:
- **lemma** — the base word
- **gap** — how far behind competitors you are (higher = more urgent)
- **coverage_percent** — share of competitor pages where this word appears in a strong zone  
- **recommendation** — concrete action (Title, H1–H3, Body)
- **context_snippet** — example phrase from competitor pages

In [ ]:
gaps = result.get("semantic_context_analysis", [])
df_gaps = pd.DataFrame(gaps)

if not df_gaps.empty:
    cols = ["lemma", "gap", "coverage_percent", "recommendation", "context_snippet"]
    cols = [c for c in cols if c in df_gaps.columns]
    df_top = df_gaps.sort_values("gap", ascending=False).head(15)
    display(df_top[cols])
else:
    print("No semantic gaps detected — excellent job!")

## § 4 — Anchors  *(Link texts competitors use, with href values)*

Unihra now returns the **exact `href` URLs** behind each anchor text. Use this to:
1. See which pages competitors link to from important keywords.
2. Identify internal-linking opportunities on your page.
3. Spot missing anchor patterns entirely absent from your page.

In [ ]:
anchors = result.get("anchors_analysis", [])
df_anchors = pd.DataFrame(anchors)

if not df_anchors.empty:
    # Highlight anchors completely absent from your page
    missing = df_anchors[df_anchors["frequency_own"] == 0].head(15)
    display(Markdown("### 🪝 Missing anchors on your page (highest competitor avg frequency)"))
    display(missing.sort_values("frequency_comp_avg", ascending=False))

    # ——— Show links for top anchor ———
    if not missing.empty and "links" in missing.columns:
        top_anchor = missing.iloc[0]
        display(Markdown(f"#### Links for  **{top_anchor['anchor']}**"))
        links = top_anchor.get("links", [])
        if links:
            display(pd.Series(links, name="href").to_frame())
        else:
            print("(no href links collected for this anchor)")
else:
    print("No anchor data returned.")

## § 5 — TF‑IDF Word Actions

Words that need to be **added** or **increased** on your page.

In [ ]:
df_words = client.get_dataframe(result, section="block_comparison")

if not df_words.empty:
    actions = df_words[df_words["action_needed"].isin(["add", "increase"])].head(15)
    display(actions[["word", "frequency", "pct_target_comp_avg", "action_needed"]])
else:
    print("No word-action data.")

## § 6 — Phrases (N‑grams)

Stable 2–3 word competitor patterns you should consider weaving into your copy.

In [ ]:
ngrams_raw = result.get("ngrams_analysis") or result.get("n_grams_analysis") or []
df_ngrams = pd.DataFrame(ngrams_raw)

if not df_ngrams.empty:
    # Show top phrases used on ≥2 competitor pages but missing from yours
    missing_ngrams = df_ngrams[
        (df_ngrams["present_on_own_page"] == False) &
        (df_ngrams["pages_count"] >= 2)
    ].sort_values("pages_count", ascending=False).head(12)
    display(missing_ngrams[["ngram", "ngram_type", "pages_count", "frequency_avg"]])
else:
    print("No n‑gram data.")

## § 7 — Triplets (Knowledge Graph) 🆕

Available only when this analysis was launched with `triplet_analysis=True` (extended mode, 5 credits).

Two views are surfaced:

1. **Entities** — the most-confirmed subjects across competitor sources, with their tier (`core` → `main` → `additional` → `unique`) and a sample of `(predicate, object, sources)` facts.
2. **Topical gaps** — subjects **absent from your page**, grouped by how many competitor sources cover them: `critical` (3+ sites), `important` (2 sites), `unique` (1 site).

In [ ]:
triplets = result.get("triplets_analysis", {})

if not triplets:
    display(Markdown(
        "⚠️  No triplets in the result — re-run § 2 with `ENABLE_TRIPLETS = True` "
        "(cost: 5 credits) to populate the Knowledge Graph."
    ))
else:
    stats = triplets.get("stats", {})
    display(Markdown(
        "### 📊 Knowledge Graph stats\n"
        f"- Total triplets: **{stats.get('total_triplets', 0)}**\n"
        f"- Sources with content: **{stats.get('sources_with_content', 0)}**\n"
        f"- Topical gaps — critical: **{stats.get('gaps_critical', 0)}**, "
        f"important: **{stats.get('gaps_important', 0)}**, "
        f"unique: **{stats.get('gaps_unique', 0)}**, "
        f"total: **{stats.get('gaps_total', 0)}**"
    ))

    # ——— Top entities (flat one-row-per-fact table) ———
    df_entities = client.get_dataframe(result, section="triplets_analysis")
    if not df_entities.empty:
        priority = df_entities[df_entities["tier"].isin(["core", "main"])]
        display(Markdown("### 🧠 Core / main entities and their facts"))
        display(priority.head(20)[["subject", "tier", "predicate", "object", "sources"]])

    # ——— Topical gaps (what your page is missing) ———
    df_gaps_t = client.get_dataframe(result, section="triplets_gaps")
    if not df_gaps_t.empty:
        display(Markdown("### 🚧 Topical gaps absent from your page"))
        order = {"critical": 0, "important": 1, "unique": 2}
        df_gaps_t = df_gaps_t.assign(_order=df_gaps_t["severity"].map(order)).sort_values("_order").drop(columns="_order")
        display(df_gaps_t.head(20))

## § 8 — Page Structure & Heading Hierarchy

Compare meta titles, H1s, content volume, and the full H1–H6 outline across all pages.

In [ ]:
structures = result.get("page_structure", [])

if structures:
    flat = []
    for page in structures:
        is_mine = page["url"] == own_page
        flat.append({
            "Page": "🟢  YOUR" if is_mine else "🔴  COMP",
            "URL": page["url"],
            "Meta Title": (page.get("meta_tags", {}) or {}).get("title", "")[:70],
            "H1": (page.get("content", {}) or {}).get("h1_heading", ""),
            "Chars (no spaces)": (page.get("metrics", {}) or {}).get("char_count_no_spaces"),
            "Uniqueness %": (page.get("metrics", {}) or {}).get("uniqueness_percentage"),
        })
    df_struct = pd.DataFrame(flat)
    display(df_struct)

    # ——— Heading tree for each page ———
    for page in structures:
        label = "🟢  YOUR" if page["url"] == own_page else "🔴  COMP"
        raw = (page.get("content", {}) or {}).get("heading_structure_raw", "")
        if raw:
            headers = raw.split("; ")
            print(f"\n{label} — {page['url']}")
            for h in headers[:12]:
                print(f"  ├ {h}")
            if len(headers) > 12:
                print(f"  └ … +{len(headers)-12} more")
else:
    print("⚠️  No structure data returned.")

## § 9 — Export to Excel

Generates a multi-sheet `.xlsx` with colour coding and auto-widths for easy sharing. When the analysis ran with triplets, the report includes additional **Triplets** and **Triplets Gaps** sheets.

In [ ]:
from datetime import datetime

filename = f"unihra_report_{datetime.now():%Y-%m-%d}.xlsx"
client.save_report(result, filename, style_output=True)
display(Markdown(f"📥  Report saved →  **{filename}**"))

---

<div align="center">
<b>Next steps</b><br>
• Replace the demo URLs with your own page and competitors.<br>
• Adjust <code>target_queries</code> to match the keywords you actually want to rank for.<br>
• Toggle <code>ENABLE_TRIPLETS</code> in §2 to switch between standard (1 credit) and Knowledge Graph (5 credits) modes.<br>

🔗  [unihra.ru](https://unihra.ru) · [API docs](https://unihra.ru/docs)
</div>